In [3]:
calendar_csv = "calendar.csv"
listing_csv = "listings.csv"

In [25]:
import pandas as pd
from data_profiling import ProfileReport
import numpy as np


df_calendar = pd.read_csv(calendar_csv)
df_listing = pd.read_csv(listing_csv)
df_listing = df_listing.dropna(axis=1, how='all')

with pd.option_context('display.max_rows', None):
    nulos = df_listing.isna().sum()
    print(nulos[nulos > 0])




description                      768
picture_url                        1
host_location                   6580
host_about                     12938
bathrooms                       3441
bathrooms_text                    26
bedrooms                        5185
beds                            2583
price                           1792
price_quote_checkin_date        1338
price_quote_checkout_date       1338
price_quote_total_price         1792
price_quote_price_per_night     1792
price_quote_raw                 1338
minimum_nights                     3
maximum_nights                     3
minimum_minimum_nights             3
maximum_minimum_nights             3
minimum_maximum_nights             3
maximum_maximum_nights             3
has_availability                 195
estimated_revenue_l365d         1797
first_review                    3728
last_review                     3728
review_scores_rating            3728
review_scores_accuracy          3728
review_scores_cleanliness       3728
r

In [27]:
#print("Nulos en 'bathrooms':", df_listing['bathrooms'].isna().sum())
#print("Nulos en 'bathrooms_text':", df_listing['bathrooms_text'].isna().sum())
print(df_listing['bathrooms_text'].unique())
# bathrooms_text -> numero | tipo_baño
# bathrooms_text -> baths | shared bath | half-bath | shared half-bath

# 1. Extraer el primer número (entero o decimal)
df_listing['bathrooms_num'] = (
    df_listing['bathrooms_text']
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)
print(df_listing['bathrooms_num'].unique())


# Ver casos donde bathrooms es NaN pero bathrooms_text tiene valor
inconsistencias = df_listing[df_listing['bathrooms'].isna() & df_listing['bathrooms_text'].notna()]
print("Filas a recuperar:", len(inconsistencias))

['1 bath' '0 baths' '1 shared bath' '1.5 baths' '3 shared baths'
 '4.5 baths' 'Half-bath' '3 baths' 'Private half-bath' '5 baths'
 '1 private bath' '2.5 baths' '2 baths' '7 baths' '5 shared baths'
 '2 shared baths' '6 shared baths' nan '3.5 baths' '1.5 shared baths'
 '6 baths' '4 shared baths' 'Shared half-bath' '5.5 shared baths'
 '8 baths' '4 baths' '0 shared baths' '12 shared baths' '5.5 baths'
 '3.5 shared baths' '9 baths' '2.5 shared baths' '12 baths' '16 baths'
 '8.5 baths' '15 baths' '4.5 shared baths' '20 shared baths' '6.5 baths'
 '10 baths' '9.5 baths' '7 shared baths' '7.5 baths' '7.5 shared baths'
 '11 baths' '8.5 shared baths' '9 shared baths' '22 baths'
 '8 shared baths' '20 baths']
[ 1.   0.   1.5  3.   4.5  nan  5.   2.5  2.   7.   6.   3.5  4.   5.5
  8.  12.   9.  16.   8.5 15.  20.   6.5 10.   9.5  7.5 11.  22. ]
Filas a recuperar: 3421


In [65]:
# Pasar al docs esta transformación de Bathrooms + Bathrooms_texts
# 2. Asignar 0.5 a las cadenas que corresponden a un medio baño sin número numérico explícito
es_medio_banio = (
    df_listing['bathrooms_text'].str.contains('half', case=False, na=False) & 
    df_listing['bathrooms_num'].isna()
)
df_listing['bathrooms_num'] = np.where(es_medio_banio, 0.5, df_listing['bathrooms_num'])


# 3. Categorizar el tipo de baño (bathrooms_type)
condiciones = [
    df_listing['bathrooms_text'].str.contains('shared', case=False, na=False),
    df_listing['bathrooms_text'].str.contains('private', case=False, na=False)
]
elecciones = ['Shared', 'Private']

# Asigna 'Shared', 'Private' o 'Standard' si no aclara
df_listing['bathrooms_type'] = np.select(condiciones, elecciones, default='Private')

# Si la celda original era NaN, mantener el tipo como NaN
#df_listing.loc[df_listing['bathrooms_text'].isna(), 'bathrooms_num'] = np.nan
#df_listing.loc[df_listing['bathrooms_text'].notna(), df_listing['bathrooms_num'].isna()] = df_listing['bathrooms_text'].notna()


# 1. Definimos la máscara para filtrar las filas deseadas
mask = df_listing['bathrooms_text'].notna() & df_listing['bathrooms_num'].isna()

# 2. Asignamos el valor numérico extraído de 'bathrooms_text' en 'bathrooms_num'
df_listing.loc[mask, 'bathrooms_num'] = (
    df_listing.loc[mask, 'bathrooms_text']
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

#with pd.option_context('display.max_rows', None):
    #print(df_listing.loc[2])
#print(df_listing.loc[2]["description"])
#print(df_listing.loc[2]["amenities"])

#df_listing.head(10)

# Muestra los índices en formato Int64Index / Index
#print(df_listing[df_listing['bathrooms_text'].str.contains('half', case=False, na=False)].head(150))

# Muestra los índices donde la columna numérica es 0.0
#indices_half = df_listing[df_listing['bathrooms_num'] == 0.0].index

#print(indices_half)
#df_listing.loc[indices_half, ['bathrooms_text', 'bathrooms_num', 'bathrooms_type']]

# Filtro: contiene 'half' Y es igual a 0
mask_conflict = (
    df_listing['bathrooms_text'].str.contains('half', case=False, na=False) & 
    (df_listing['bathrooms_num'] == 0.0)
)

# Obtenemos los índices
indices_conflict = df_listing[mask_conflict].index

print(f"Total de casos encontrados: {len(indices_conflict)}")
print("Índices:", indices_conflict.tolist())

# Visualizar el resultado
df_listing.loc[indices_conflict, ['bathrooms_text', 'bathrooms_num', 'bathrooms_type']]

df_listing = df_listing.drop(columns=['bathrooms_text'])



KeyError: 'bathrooms_text'

In [ ]:
# Creamos máscaras booleanas para evaluar los estados
ambos_presentes = df_listing['bathrooms'].notna() & df_listing['bathrooms_num'].notna()
son_iguales = np.isclose(df_listing['bathrooms'], df_listing['bathrooms_num'], equal_nan=False)

# Caso 1: Inconsistencias (Ambas existen pero tienen valores distintos)
inconsistencias = df_listing[ambos_presentes & ~son_iguales]

# Caso 2: Recuperables (bathrooms es NaN pero bathrooms_num tiene dato)
recuperables = df_listing[df_listing['bathrooms'].isna() & df_listing['bathrooms_num'].notna()]

#with pd.option_context('display.max_rows', None):
 #   print(df_listing.loc[19202])
#print(df_listing.loc[19202]["description"])
#print(df_listing.loc[19202]["amenities"])
df_listing['bathrooms_num'] = np.maximum(df_listing['bathrooms_num'], df_listing['bathrooms'])
df_listing = df_listing.drop(columns=['bathrooms'])


### Columnas con nulos:
description                      768
picture_url                        1
host_location                   6580
host_about                     12938
bathrooms                       3441
bathrooms_text                    26
bedrooms                        5185
beds                            2583
price                           1792
price_quote_checkin_date        1338
price_quote_checkout_date       1338
price_quote_total_price         1792
price_quote_price_per_night     1792
price_quote_raw                 1338
minimum_nights                     3
maximum_nights                     3
minimum_minimum_nights             3
maximum_minimum_nights             3
minimum_maximum_nights             3
maximum_maximum_nights             3
has_availability                 195
estimated_revenue_l365d         1797
first_review                    3728
last_review                     3728
review_scores_rating            3728
review_scores_accuracy          3728
review_scores_cleanliness       3728
review_scores_checkin           3728
review_scores_communication     3728
review_scores_location          3728
review_scores_value             3728
license                        29199
reviews_per_month               3728

In [ ]:
# Pruebas de diferencia entre cantidad de columnas del dataset comparado con las columnas de los metadatos.
metadatos = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'calendar_last_scraped', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'license', 'instant_bookable', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month']
metadatos = set(metadatos)

columnas_dataset = ['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'calendar_last_scraped', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'license', 'instant_bookable', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms', 'reviews_per_month']
columnas_dataset = set(columnas_dataset)

print(metadatos.difference(columnas_dataset))

{'price_quote_price_per_night', 'hosts_time_as_host_years', 'estimated_revenue_l365d', 'hosts_time_as_user_months', 'availability_eoy', 'hosts_time_as_host_months', 'host_profile_url', 'price_quote_checkout_date', 'host_profile_id', 'price_quote_raw', 'hosts_time_as_user_years', 'price_quote_total_price', 'price_quote_checkin_date', 'estimated_occupancy_l365d', 'bathrooms', 'number_of_reviews_ly'}


In [13]:
# Borramos columnas totalmente nulas con
df_limpio = df_listing.dropna(axis=1, how='all')

#Se borrran 13 columnas
set_listing = set(df_listing.columns)
set_limpio = set(df_limpio.columns)

print(set_listing.difference(set_limpio))
# {'neighbourhood_group_cleansed', 'host_response_rate', 'host_neighbourhood', 'host_verifications', 'host_acceptance_rate', 'host_response_time', 'host_since', 'calendar_updated', 'instant_bookable', 'neighbourhood', 'neighborhood_overview', 'host_thumbnail_url', 'host_total_listings_count'}

{'neighbourhood_group_cleansed', 'host_response_rate', 'host_neighbourhood', 'host_verifications', 'host_acceptance_rate', 'host_response_time', 'host_since', 'calendar_updated', 'instant_bookable', 'neighbourhood', 'neighborhood_overview', 'host_thumbnail_url', 'host_total_listings_count'}


In [ ]:
#